# Sentiment analysis of Pfizer vaccine tweets

Around 11,000 tweets about the Pfizer/BioNTech vaccine, collected by Gabriel Preda and shared on Kaggle.

The tweets come without sentiment labels, so the plan is:

1. clean the text
2. label each tweet with TextBlob (positive / neutral / negative)
3. look at the data with a few charts
4. train a Logistic Regression on TF-IDF features to predict the TextBlob label
5. label the tweets again with VADER and see how much the two tools agree

Everything runs on CPU in well under a minute.

In [ ]:
import re
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_colwidth", 120)

IMG_DIR = Path("images")
IMG_DIR.mkdir(exist_ok=True)

ORDER = ["negative", "neutral", "positive"]
COLORS = {"negative": "#d9534f", "neutral": "#9e9e9e", "positive": "#4caf50"}

## 1. Data

Dataset: [Pfizer Vaccine Tweets](https://www.kaggle.com/datasets/gpreda/pfizer-vaccine-tweets) on Kaggle.

The cell below downloads it with `kagglehub` the first time and keeps a copy in `data/`. If the download does not work for you, get `vaccination_tweets.csv` from the Kaggle page by hand and drop it in the `data` folder.

In [ ]:
DATA_PATH = Path("data/vaccination_tweets.csv")

if not DATA_PATH.exists():
    import kagglehub
    folder = kagglehub.dataset_download("gpreda/pfizer-vaccine-tweets")
    DATA_PATH.parent.mkdir(exist_ok=True)
    shutil.copy(Path(folder) / "vaccination_tweets.csv", DATA_PATH)

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head(3)

In [ ]:
df.info()

In [ ]:
df.isna().sum()

Only `text` and `date` are needed from here on. Location, hashtags etc. have a lot of missing values anyway.

In [ ]:
df = df[["date", "text"]].copy()
df = df.dropna(subset=["text"])
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df.shape

## 2. Cleaning

Links, @mentions, numbers and punctuation go out. I keep the hashtag words (just drop the `#`) because people often put the actual opinion in them. Stop words are *not* removed here, TextBlob needs words like "not" to get negation right. The TF-IDF step removes them later.

In [ ]:
def clean_tweet(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)   # links
    text = re.sub(r"@\w+", " ", text)                # mentions
    text = re.sub(r"&\w+;", " ", text)               # html stuff like &amp;
    text = re.sub(r"[^a-z' ]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["clean_text"] = df["text"].astype(str).apply(clean_tweet)

# a few tweets are only a link or only emojis, nothing left after cleaning
df = df[df["clean_text"].str.len() > 0]

before = len(df)
df = df.drop_duplicates(subset="clean_text").reset_index(drop=True)
print("removed", before - len(df), "duplicate tweets")
print("tweets left:", len(df))

In [ ]:
df[["text", "clean_text"]].sample(5, random_state=1)

## 3. Labelling with TextBlob

TextBlob gives a polarity score between -1 and 1. Above 0 is positive, below 0 is negative, exactly 0 is neutral.

In [ ]:
def polarity_to_label(p):
    if p > 0:
        return "positive"
    if p < 0:
        return "negative"
    return "neutral"

df["polarity"] = df["clean_text"].apply(lambda t: TextBlob(t).sentiment.polarity)
df["sentiment"] = df["polarity"].apply(polarity_to_label)

df["sentiment"].value_counts()

In [ ]:
(df["sentiment"].value_counts(normalize=True) * 100).round(2)

## 4. Looking at the data

In [ ]:
counts = df["sentiment"].value_counts().reindex(ORDER)

plt.figure(figsize=(6, 4))
bars = plt.bar(counts.index, counts.values, color=[COLORS[s] for s in counts.index])
plt.bar_label(bars)
plt.title("Tweets per sentiment (TextBlob)")
plt.ylabel("number of tweets")
plt.savefig(IMG_DIR / "01_sentiment_counts.png", bbox_inches="tight")
plt.show()

A lot of tweets are news style ("X country received Y doses"). TextBlob finds no opinion word in those, so polarity is exactly 0 and they end up neutral. The histogram below shows that spike at zero.

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(df["polarity"], bins=40, color="#4a7ab5", edgecolor="white")
plt.title("TextBlob polarity scores")
plt.xlabel("polarity")
plt.ylabel("number of tweets")
plt.savefig(IMG_DIR / "02_polarity_hist.png", bbox_inches="tight")
plt.show()

In [ ]:
weekly = (
    df.dropna(subset=["date"])
      .groupby([pd.Grouper(key="date", freq="W"), "sentiment"])
      .size()
      .unstack(fill_value=0)
      .reindex(columns=ORDER, fill_value=0)
)

weekly.plot(figsize=(10, 4), color=[COLORS[s] for s in ORDER], linewidth=2)
plt.title("Tweets per week by sentiment")
plt.xlabel("")
plt.ylabel("number of tweets")
plt.legend(title="")
plt.savefig(IMG_DIR / "03_weekly_sentiment.png", bbox_inches="tight")
plt.show()

In [ ]:
df["n_words"] = df["clean_text"].str.split().str.len()

plt.figure(figsize=(7, 4))
bins = range(0, df["n_words"].max() + 2, 2)
for label in ORDER:
    plt.hist(df.loc[df["sentiment"] == label, "n_words"], bins=bins, histtype="step",
             linewidth=2, density=True, color=COLORS[label], label=label)
plt.title("Tweet length by sentiment")
plt.xlabel("words per tweet")
plt.ylabel("share of tweets")
plt.legend()
plt.savefig(IMG_DIR / "04_length_by_sentiment.png", bbox_inches="tight")
plt.show()

df.groupby("sentiment")["n_words"].mean().round(1)

Most common words in each group. Words like "vaccine" or "pfizer" are in nearly every tweet so I leave them out, otherwise the three plots look the same.

In [ ]:
extra_stop = ["pfizerbiontech", "pfizer", "biontech", "vaccine", "vaccines", "covid", "covidvaccine",
              "coronavirus", "amp", "dose", "doses", "vaccination", "vaccinated", "covid19"]
stop_words = list(ENGLISH_STOP_WORDS) + extra_stop

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

for ax, label in zip(axes, ORDER):
    cv = CountVectorizer(stop_words=stop_words, max_features=2000)
    mat = cv.fit_transform(df.loc[df["sentiment"] == label, "clean_text"])
    freq = pd.Series(np.asarray(mat.sum(axis=0)).ravel(), index=cv.get_feature_names_out())
    top = freq.sort_values(ascending=False).head(12)[::-1]
    ax.barh(top.index, top.values, color=COLORS[label])
    ax.set_title(label)

fig.suptitle("Most frequent words per sentiment")
plt.tight_layout()
plt.savefig(IMG_DIR / "05_top_words.png", bbox_inches="tight")
plt.show()

## 5. Logistic Regression

TF-IDF on unigrams and bigrams, capped at 5000 features so it stays light. 80/20 split, stratified so the test set has the same class mix.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"], df["sentiment"],
    test_size=0.2, random_state=42, stratify=df["sentiment"]
)

tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), max_features=5000)
X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

print(X_train_vec.shape, X_test_vec.shape)

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)

acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {acc * 100:.2f}%")
print()
print(classification_report(y_test, y_pred, digits=3))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=ORDER)

plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=ORDER, yticklabels=ORDER, cbar=False)
plt.title("Confusion matrix (test set)")
plt.xlabel("predicted")
plt.ylabel("TextBlob label")
plt.savefig(IMG_DIR / "06_confusion_matrix.png", bbox_inches="tight")
plt.show()

Words the model leans on most for the positive and negative class:

In [ ]:
words = tfidf.get_feature_names_out()
classes = list(model.classes_)

for label in ["positive", "negative"]:
    coefs = model.coef_[classes.index(label)]
    top = np.argsort(coefs)[-15:][::-1]
    print(label, "->", ", ".join(words[top]))
    print()

These are basically the words from TextBlob's own lexicon, which makes sense. The model is learning to copy TextBlob, so the accuracy here means "agreement with TextBlob", not agreement with what a human reader would say.

## 6. Comparison with VADER

VADER is another rule based tool, built for social media text. It looks at capital letters, exclamation marks and emojis, so I give it the original tweet (minus links) instead of the cleaned one. Usual thresholds: compound >= 0.05 positive, <= -0.05 negative.

In [ ]:
vader = SentimentIntensityAnalyzer()

def vader_label(text):
    text = re.sub(r"http\S+", "", text)
    score = vader.polarity_scores(text)["compound"]
    if score >= 0.05:
        return "positive"
    if score <= -0.05:
        return "negative"
    return "neutral"

df["vader_sentiment"] = df["text"].astype(str).apply(vader_label)

compare = pd.DataFrame({
    "TextBlob": df["sentiment"].value_counts(),
    "VADER": df["vader_sentiment"].value_counts(),
}).reindex(ORDER)
compare

In [ ]:
agreement = (df["sentiment"] == df["vader_sentiment"]).mean()
print(f"TextBlob and VADER give the same label for {agreement * 100:.2f}% of tweets")

In [ ]:
ct = pd.crosstab(df["sentiment"], df["vader_sentiment"], normalize="index").reindex(index=ORDER, columns=ORDER, fill_value=0) * 100

plt.figure(figsize=(5.5, 4.5))
sns.heatmap(ct, annot=True, fmt=".1f", cmap="Greens", cbar=False)
plt.title("TextBlob vs VADER (% of each TextBlob class)")
plt.xlabel("VADER label")
plt.ylabel("TextBlob label")
plt.savefig(IMG_DIR / "07_textblob_vs_vader.png", bbox_inches="tight")
plt.show()

How does the trained model do if VADER labels are treated as the truth on the same test tweets?

In [ ]:
vader_test = df.loc[X_test.index, "vader_sentiment"]
print(f"Model vs TextBlob labels: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(f"Model vs VADER labels:    {accuracy_score(vader_test, y_pred) * 100:.2f}%")

Some tweets where the two tools disagree completely:

In [ ]:
opposite = df[
    ((df["sentiment"] == "positive") & (df["vader_sentiment"] == "negative"))
    | ((df["sentiment"] == "negative") & (df["vader_sentiment"] == "positive"))
]
print(len(opposite), "tweets with opposite labels")
opposite[["text", "sentiment", "vader_sentiment"]].sample(min(8, len(opposite)), random_state=3)

## Notes

- Labels come from TextBlob, not from people. So the Logistic Regression score tells how well a simple model can reproduce TextBlob, nothing more.
- Both tools are word-list based, so sarcasm and vaccine slang ("shot", "jab") can easily get the wrong label.
- A small hand labelled sample would be the obvious next step to check which tool is closer to reality.